### ```__str__```

The method ```__str__``` is called when the print statement needs to print the value of an instance. It returns a string. The print-format expression calls this for conversion ```%s```.

This is a subtlety worth slowing down on, because it maps directly to your binary mode ```b'...'``` display discussion earlier!

In [1]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __str__(self):
        return f"({self.x}, {self.y})"           # friendly, for humans

    def __repr__(self):
        return f"Point({self.x}, {self.y})"        # precise, could recreate the object

In [2]:
p = Point(3, 4)

print(p)          # → (3, 4)              calls __str__  — used by print()
p                 # → Point(3, 4)          calls __repr__ — used when Jupyter/REPL  displays the value directly

(3, 4)


Point(3, 4)

```__str__``` — for ```print()```. Meant to be readable for a human.

## Understanding "The print-format expression calls this for conversion `%s`"

This is referring back to the **oldest** formatting style you learned — the `%` syntax — and pointing out that its `%s` specifier is secretly powered by `__str__`.

---

### Recall — The `%` Format Syntax

Way back, you learned three formatting styles:

```python
"%i plus %i is equal to %i" % (1, 3, 4)     # % syntax
"{} plus {} is equal to {}".format(1, 3, 4) # .format() method
f"{1} plus {3} is equal to {4}"              # f-string
```

You also learned `%i` for integers, `%f` for floats. There's a **third** common specifier in the `%` family: **`%s`** — meaning *"insert this as a string."*

```python
name = "Alice"
print("Hello, %s!" % name)
# → Hello, Alice!
```

---

### What `%s` Actually Does — Converts ANYTHING to a String

`%s` isn't picky about type — it accepts **any object** and converts it to text for you:

```python
print("Value: %s" % 42)        # → Value: 42       (int → string)
print("Value: %s" % [1,2,3])    # → Value: [1, 2, 3]  (list → string)
print("Value: %s" % 3.14)        # → Value: 3.14       (float → string)
```

**The question is: HOW does `%s` know how to turn any of these into text?** That's exactly what your quoted sentence answers.

---

### The Connection — `%s` Calls `__str__` Behind the Scenes

```python
"%s" % obj
```

is really doing:

```python
obj.__str__()
```

So when you write `%s`, Python's formatting machinery internally calls the object's own `__str__` method to get its text representation — the **same** method that `print()` uses!

```python
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __str__(self):
        return f"({self.x}, {self.y})"

p = Point(3, 4)

print(p)               # → (3, 4)     print() calls __str__
print("Point: %s" % p)  # → Point: (3, 4)     %s ALSO calls __str__!
```

Both `print(p)` and `"%s" % p` land on the **exact same method** — `p.__str__()` — just triggered through two different pieces of syntax.

---

### Why This Matters — Consistency Across the Language

This is the deeper point of the whole "special methods" section: Python doesn't have **separate, unrelated** mechanisms for "how print shows an object" vs "how `%s` formatting shows an object." They're **unified** — both roads lead to the same `__str__` method you define once on your class.

```
print(obj)          ─┐
"%s" % obj            ├──►  ALL call obj.__str__()
f"{obj}"                │      (f-strings and .format() also route through __str__!)
"{}".format(obj)     ─┘
```

Define `__str__` **once**, and every formatting style in Python automatically knows how to display your object sensibly — you don't need a separate method for each formatting style.

---

### Proving f-strings Use It Too

```python
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __str__(self):
        return f"({self.x}, {self.y})"

p = Point(3, 4)

print(f"My point: {p}")          # → My point: (3, 4)
print("My point: {}".format(p))  # → My point: (3, 4)
print("My point: %s" % p)        # → My point: (3, 4)
```

All three lines — f-string, `.format()`, and `%s` — produce **identical** output, because all three ultimately call the **same** `__str__` method under the hood.

---

### Without `__str__` — The Default Fallback

If you don't define `__str__`, Python falls back to a generic, not-very-useful default:

```python
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

p = Point(3, 4)
print("%s" % p)
# → <__main__.Point object at 0x7f8b2c3d4f10>     ← unhelpful memory address!
```

This is exactly why defining `__str__` is worthwhile — without it, **every** formatting style (print, `%s`, f-strings) shows this ugly default instead of something meaningful.

---

### The One-Sentence Summary

> The sentence means: when you use the old `%` formatting style with a `%s` placeholder (`"... %s ..." % obj`), Python converts `obj` to text by calling its `__str__()` method — the **exact same method** that `print(obj)` uses. This is Python's way of making sure all its formatting tools (`%s`, f-strings, `.format()`, `print()`) stay consistent, since they all funnel through the one `__str__` method you define on your class. 🎯

### ```__repr__```

The method ```__repr__``` is called when the interactive interpreter prints the value of an evaluated expression, and when the conversion ```%r``` for print-format expression is used. Returns a canonical representation string that (at least in theory) can be used to recreate the original object.

```__repr__``` — for the interactive shell auto-display and debugging. Meant to be precise — ideally, text you could paste back into Python to recreate the exact same object.

This mirrors what you discovered with bytes: ```repr(bytes_object)``` showed ```b'Hello\r\nWorld'``` — a precise, technically-accurate display — while print() of a plain string shows the friendly, human-facing version. Same ```__str__/__repr__``` split, just built into Python's own types already.

If ```__str__``` is missing, Python falls back to using ```__repr__``` for ```print()``` too — so defining ```__repr__``` alone still gives you something reasonable.

## `__repr__` — Simply

Same idea as `__str__`, but for a **different** situation: when Python needs to show a value **automatically**, without you explicitly calling `print()`.

---

### The First Trigger — "The Interactive Interpreter Prints the Value of an Evaluated Expression"

Remember way back — your very first lesson on expressions vs statements? You learned:

> *"In the interactive Python shell (REPL), bare expressions ARE automatically displayed — but in a script, they are silently discarded."*

That auto-display behavior is **exactly** what this sentence is about — and it's powered by `__repr__`.

```python
>>> "hello"
'hello'
```

You didn't call `print()`. You just typed an expression, hit Enter, and Jupyter/the REPL **automatically showed you the value**. That automatic display is calling `__repr__` behind the scenes:

```python
>>> "hello".__repr__()
"'hello'"
```

---

### Watch `__str__` and `__repr__` Diverge — Same Object, Different Trigger

```python
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __str__(self):
        return f"({self.x}, {self.y})"        # for print()

    def __repr__(self):
        return f"Point({self.x}, {self.y})"     # for auto-display / %r

p = Point(3, 4)
```

```python
print(p)     # → (3, 4)              ← calls __str__
p            # → Point(3, 4)          ← typed bare in REPL/Jupyter, calls __repr__!
```

**Same object, two different-looking outputs**, depending on **how** you're viewing it — explicitly printed vs. auto-displayed by the interpreter.

---

### Why Have Two Separate Methods At All?

Because they serve **different audiences**:

| | `__str__` | `__repr__` |
|---|---|---|
| Triggered by | `print()`, `%s`, f-strings | typing a bare expression in REPL/Jupyter, `%r`, `repr()` |
| Meant for | a human reading casual output | debugging — precise, unambiguous |
| Goal | readable, friendly | *"could I paste this back into Python to recreate the object?"* |

The exercise text calls `__repr__`'s output a **"canonical representation... that can be used to recreate the original object"** — meaning `Point(3, 4)` isn't just descriptive text, it's **valid Python code** you could literally run to build an identical object again:

```python
eval(repr(p))    # → creates a NEW Point(3, 4), from the repr string!
```

`(3, 4)` (the `__str__` version) couldn't be executed as code the same way — it's just for a human's eyes.

---

### The Second Trigger — `%r` in Format Strings

Just like `%s` calls `__str__`, there's a matching **`%r`** that calls `__repr__` instead:

```python
p = Point(3, 4)

print("Value: %s" % p)     # → Value: (3, 4)          uses __str__
print("Value: %r" % p)     # → Value: Point(3, 4)      uses __repr__
```

Same object — `%s` gives you the friendly version, `%r` gives you the precise, code-like version. The `s`/`r` letters are just mnemonics: **s**tring vs **r**epresentation.

---

### You've Actually SEEN This Distinction Already!

Remember your very own bytes lesson —

```python
b = b'Hello\r\nWorld'
```

That `b'Hello\r\nWorld'` display **IS** `bytes`'s `__repr__` in action — showing you a precise, code-paste-able form (including the `b'...'` prefix, and `\r`/`\n` written out explicitly) rather than a plain human-friendly rendering. Same underlying mechanism, different type.

And even ordinary strings show this split constantly:

```python
s = "hello"
print(s)      # → hello           ← __str__ (no quotes, just the content)
s              # → 'hello'          ← __repr__ (quotes included — precise, code-like)
```

---

### What If You Only Define `__repr__`, Not `__str__`?

Handy fallback rule: if a class has `__repr__` but **no** `__str__`, Python uses `__repr__` for **both** situations:

```python
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __repr__(self):
        return f"Point({self.x}, {self.y})"
    # no __str__ defined!

p = Point(3, 4)
print(p)     # → Point(3, 4)     ← falls back to __repr__, since __str__ is missing
```

This is why many simple classes only bother defining `__repr__` — it covers both cases reasonably well, whereas defining only `__str__` leaves the REPL/`%r` case showing the ugly default `<Point object at 0x...>`.

---

### The One-Sentence Summary

> `__repr__` controls how an object appears when Python **automatically** displays it — typing a bare expression in the interactive shell/Jupyter, or using the `%r` format specifier — as opposed to `__str__`, which controls the output of an **explicit** `print()` call or `%s`. `__repr__` is meant to be a precise, ideally code-recreatable representation (like `Point(3, 4)`), while `__str__` is meant to be friendly and readable (like `(3, 4)`) — the same "precise vs friendly" split you already saw with bytes' `b'...'` display versus a plain string's printed content. 🎯